# IOMEval Project Context

## Overview

**IOMEval** is a project of IOM's (International Organization for Migration) Central Evaluation Division (CED) for processing and tagging IOM evaluation reports against international migration frameworks. It comprises two main components:

- **IOMEval** (`iomeval/`) — a Python library (nbdev) for the end-to-end processing pipeline
- **TagApp** (`tagapp/`) — a FastHTML web app (deployed to pla.sh) for CED evaluators and staff to review and edit AI-generated tags

## Pipeline: Download → OCR → Curate → Map → Deploy

1. **Data ingestion** (`readers.py`): IOM evaluation repository CSV → normalized `evaluations.json`, enriched with report URLs
2. **Download** (`downloaders.py`): Fetch PDF reports → `data/pdf/{eval_id}/`
3. **OCR** (via [`mistocr`](https://github.com/answerdotai/mistocr), an external OCR library): PDF → markdown with heading hierarchy fixing → `data/md/{eval_id}/`
4. **Curation** (`curator.py`): FastHTML app to (i) fix markdown heading hierarchy manually, (ii) select sections/subsections for tagging
5. **Mapping** (`mapper.py`): LLM-based tagging (via `litellm`) of selected sections against:
   - **GCM** objectives (Global Compact for Migration)
   - **SRF outputs** (Strategic Results Framework)
   - **SRF enablers**
   - **Cross-cutting priorities** (CCPs)
6. **Results** → `data/results/{eval_id}.json` (one JSON per report with metadata, sections, mappings)
7. **Deploy** → copy new results to `data/results_new/`, deploy to TagApp via `tagapp/nbs/08_ops.ipynb`

## Pipeline steps (from `pipeline.py`)

The `Report` class provides a **chainable API** — each method returns `self`:

1. `Report.from_url()` / `.from_title()` / `.from_id()` — create from evaluation repository
2. `.download()` — fetch PDF → `data/pdf/{id}/`
3. `.ocr()` — PDF → markdown (via `mistocr`) → `data/md/{id}/`
4. *(manual)* Curate in curator FastHTML app → sets `curation_status='sections_selected'`
5. `.map_enbs()` — map to SRF enablers
6. `.map_ccps()` — map to cross-cutting priorities
7. `.map_gcms()` — map to GCM objectives
8. `.map_outs()` — map to SRF outputs (filtered by top GCM matches)
9. `.save()` — called automatically after each step

`run_pipeline()` orchestrates all steps for a single report.
`batch_run()` processes multiple reports. It **skips** reports already completed (all 4 mapping types present) unless `force` is set. Reports not yet curated (`curation_status != 'sections_selected'`) are returned as `awaiting_curation`. It can be filtered by `year` or `ids`. Results are tracked as completed / awaiting_curation / failed / skipped and saved to `data/batch_runs/`.

## Result JSON structure

Each report produces a single JSON file (`data/results/{eval_id}.json`) that serves as both a **status file** and a **results file**. It is progressively filled as the pipeline advances:

```json
{
  "id": "md5_hash",                 // Evaluation ID (MD5 hash)
  "report_url": "https://...",      // Source PDF URL
  "meta": {                         // Evaluation metadata from IOM repository
    "Title": "...",
    "Year": 2023,
    "Countries Covered": [...],
    "Evaluation Commissioner": "...",
    // ... ~25 metadata fields
  },
  "docs": [...],                    // Available document links
  "curation_status": "pending|sections_selected",
  "selected_headings": [            // Headings chosen in curator app
    "## EXECUTIVE SUMMARY ... page 12",
    "## 5. CONCLUSIONS ... page 69"
  ],
  "mappings": {                     // LLM-generated theme mappings (added progressively)
    "enbs": [{...}, ...],           // SRF enablers
    "ccps": [{...}, ...],           // Cross-cutting priorities
    "gcms": [{...}, ...],           // GCM objectives
    "outs": [{...}, ...]            // SRF outputs
  },
  "timestamp": "ISO datetime"       // Last save timestamp
}
```

### Mapping output structure

Each item in a mapping array has:

```json
{
  "theme_id": "1",
  "theme_title": "Workforce",
  "relevance_score": 0.28,
  "reasoning": "Detailed explanation of relevance assessment..."
}
```

### Progressive pipeline & forcing

The JSON file is saved after each pipeline step (OCR, curation, each mapping type). Each step checks if its output already exists and **skips** if so. The `force` argument overrides this:

- `force=True` — re-runs all steps
- `force={'ocr', 'enbs'}` — re-runs only named steps (via `should_force()`)

This allows selective re-processing (e.g. re-mapping with a different model) without re-downloading or re-OCRing.

## Key modules (`iomeval/`)

- `core.py` — utility functions (token counting, prompt loading, ...)
- `readers.py` — evaluation repository loading, normalization, URL resolution (`eval_url`, `find_eval`)
- `downloaders.py` — PDF download from IOM evaluation repository
- `extract.py` — section extraction from markdown (heading hierarchy utilities)
- `curator.py` — FastHTML curation app for heading fixing and section selection
- `themes.py` — load/format GCM, SRF enablers, CCPs, and SRF outputs theme definitions
- `mapper.py` — LLM mapping assembly (system blocks, prompts, caching, via `litellm`)
- `pipeline.py` — `Report` class (chainable), `run_pipeline`, `batch_run`

## Data layout (`iomeval/data/`)

- `pdf/` — downloaded PDFs (by eval_id)
- `md/` — OCR'd markdown (by eval_id)
- `results/` — JSON results per evaluation (metadata + sections + mappings)
- `results_new/` — staged results for TagApp deployment
- `batch_runs/` — batch processing logs

## TagApp (`tagapp/`)

- FastHTML web app for CED evaluators to review and edit AI-generated tags
- SQLite DB + JSON data files per report
- Deployed to pla.sh via `tagapp/nbs/08_ops.ipynb`
- Backups stored in `tagapp/backups/` on each deployment
- New results are staged in `iomeval/data/results_new/` and copied during deployment

## Key dialogs

- `nbs/analysis/processing` — main working dialog for batch processing and analysis
- `nbs/07_pipeline.ipynb` — pipeline module development (nbdev)
- `tagapp/nbs/08_ops.ipynb` — deployment operations

## Scripts (`scripts/`)

- `bu_prompts` — backs up current prompt `.md` files from `nbs/files/prompts/` into `nbs/files/prompts/legacy/` with a date suffix (ddmmyy)
- `sync_prompts` — copies prompt `.md` files from `nbs/files/prompts/` to the exported `iomeval/files/prompts/` folder
- `sync_themes` — copies theme files from `nbs/files/themes/` to the exported `iomeval/files/themes/` folder

## Operations Checklists

### Updated prompts
1. Run `scripts/bu_prompts` (backup to `legacy/`)
2. Edit prompts in `nbs/files/prompts/`
3. Run `scripts/sync_prompts` (copy to `iomeval/files/prompts/`)
4. Re-run mappings with `force={'enbs','ccps','gcms','outs'}` (or subset) on target report IDs
5. Compare results (old backup vs new `data/results/`) — review in analysis dialog
6. Copy updated results to `data/results_new/`
7. Deploy via `tagapp/nbs/08_ops.ipynb`

### Updated themes
1. Edit themes in `nbs/files/themes/`
2. Run `scripts/sync_themes` (copy to `iomeval/files/themes/`)
3. Re-run affected mappings with `force`
4. Compare results and review

### Deploy to TagApp
1. Stage new results to `data/results_new/`
2. Deploy via `tagapp/nbs/08_ops.ipynb`

How to say now "